In [394]:
# Standard libraries
import os
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, mutual_info_regression, f_classif, f_regression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Other utilities
from scipy.stats import randint, uniform
import joblib

# Standard libraries
from typing import List, Tuple

# Data manipulation and numerical computation
import numpy as np
import pandas as pd

# Scikit-learn utilities
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold, f_classif, f_regression

In [395]:
path = "../../data/raw/"
dfs = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs[name].shape}")

Loaded 01_DiatomInventories_GTstudentproject_B with shape (1643872, 8)
Loaded 02_InfoSites_GTstudentproject_B with shape (8404, 11)
Loaded 03_IBD_GTstudentproject_test with shape (5063, 2)
Loaded 03_IBD_GTstudentproject_train with shape (43783, 4)
Loaded 04_PressureStatus_GTstudentproject_B with shape (49231, 30)
Loaded 05_EnvParamMeans_GTstudentproject_B with shape (3763903, 8)
Loaded 06_ListEnvParam_GNNprojectGT_B with shape (192, 14)
Loaded 07_TaxaCode_GTstudentproject_B with shape (2292, 2)


In [396]:
taxones = dfs[list(dfs.keys())[0]] 

In [397]:
pressure = dfs[list(dfs.keys())[4]]

In [398]:
path = "../../data/processed/"
dfs_processed = {}

# read all dataframes and keep them in dfs
for parquet in os.listdir(path):  # List all files in the directory
    if parquet.endswith(".parquet"):
        name = parquet.split(".parquet")[0]  # Get name without extension
        dfs_processed[name] = pd.read_parquet(os.path.join(path, parquet))  # Use os.path.join for paths
        print(f"Loaded {name} with shape {dfs_processed[name].shape}")

Loaded 03_CLEAN_COMPLETE_DF with shape (49863, 174)
Loaded 03_CLEAN_COMPLETE_DF_02 with shape (49231, 623)
Loaded 03_COMPLETE_TEST with shape (43568, 3)
Loaded 03_COMPLETE_TRAIN with shape (49441, 2332)
Loaded 03_COMPLETE_TRAIN_2 with shape (49231, 2849)
Loaded clean_train with shape (43568, 4)
Loaded dep_codes with shape (49231, 12)
Loaded dep_test with shape (5063, 12)
Loaded taxones_pressure with shape (5663, 2333)
Loaded taxones_pressure_epm_predict with shape (5663, 2850)
Loaded taxones_pressure_epm_train with shape (43568, 2853)
Loaded taxones_pressure_predict with shape (5663, 2333)
Loaded taxones_pressure_train with shape (43568, 2336)


In [399]:
sites = dfs_processed[list(dfs_processed.keys())[6]]

In [400]:
train = dfs_processed[list(dfs_processed.keys())[5]]

In [401]:
test = dfs_processed[list(dfs_processed.keys())[7]]

In [402]:
test_codes = test['CodeDepartement'].unique()

In [403]:
regiones = sites['HERlvl1Code'].drop_duplicates().tolist()
len(regiones)

22

ejemplo antes de volverlo función

In [404]:
region = 21

df = sites[sites['HERlvl1Code']== region]

valid_codes = df ['SamplingOperations_code'].unique()

df1 = taxones[taxones['SamplingOperations_code'].isin(valid_codes)]

# 1) columnas que QUIERES mantener en la llave (todas menos: TaxonName y Abundance_nbcell;
#    y quitamos TaxonCode/Abundance_pm porque se usan para columnas/valores)
cols_key = [c for c in df1.columns 
            if c not in ['TaxonName', 'Abundance_nbcell', 'TaxonCode', 'Abundance_pm']]

# 2) tabla pivote:
#    - index = todas las columnas que quieres conservar (parte del key)
#    - columns = códigos de taxón
#    - values = Abundance_pm
#    - aggfunc='sum' por si hay duplicados del mismo taxón en la misma muestra
wide = (
    df1.pivot_table(index=cols_key,
                   columns='TaxonCode',
                   values='Abundance_pm',
                   aggfunc='sum')
      .reset_index()
)

# opcional: quitar el nombre del eje de columnas
wide.columns.name = None

df2 = wide

df2 = pd.merge(wide, pressure, on=['SamplingOperations_code', 'CodeSite_SamplingOperations','Date_SamplingOperation'],how = 'inner')

train_df = pd.merge(df2, train, on= 'SamplingOperations_code',how = 'inner')
test_df = df2[df2['SamplingOperations_code'].isin(test_codes)]


In [405]:
def drop_exact_duplicates(X: pd.DataFrame) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop exact duplicate columns from a DataFrame.

    This function identifies and removes columns in the DataFrame that are exact duplicates 
    of other columns. Duplicate columns are those that have identical values across all rows.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with duplicate columns removed.
        - A list of the names of the dropped duplicate columns.

    Notes:
    -----
    - The function computes a hash signature for each column to efficiently identify duplicates.
    - If two columns have the same hash and their values are identical, one of them is dropped.

    Example Usage:
    --------------
    X, dropped_dup_cols = drop_exact_duplicates(X)
    print(f"Dropped exact duplicate columns: {dropped_dup_cols}")
    """
    sig = X.apply(lambda s: pd.util.hash_pandas_object(s, index=False).sum())
    seen = {}
    dup = []
    for c, h in sig.items():
        if h in seen and X[c].equals(X[seen[h]]):
            dup.append(c)
        else:
            seen[h] = c
    return X.drop(columns=dup), dup

def drop_high_missing(X: pd.DataFrame, thresh: float = 0.40) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop columns with a high proportion of missing values from a DataFrame.

    This function identifies columns in the DataFrame where the proportion of missing values 
    exceeds a specified threshold and removes them. Missing values are identified as NaN 
    or the string "Unassessed", which is replaced with NaN before processing.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    thresh : float, optional
        The proportion threshold above which a column is considered to have high missing values 
        and is dropped. Default is 0.40 (40%).

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-missing columns removed.
        - A list of the names of the dropped columns.

    Notes:
    -----
    - The function replaces the string "Unassessed" with NaN before calculating the proportion 
      of missing values.
    - Columns with a proportion of missing values greater than `thresh` are dropped.

    Example Usage:
    --------------
    X, dropped_cols = drop_high_missing(X, thresh=0.40)
    print(f"Dropped columns with >40% missing: {dropped_cols}")
    """
    # Replace "Unassessed" with NaN
    X = X.replace("Unassessed", np.nan)

    # Identify columns with a high proportion of missing values
    to_drop = X.columns[X.isna().mean() > thresh].tolist()

    # Drop the identified columns
    X2 = X.drop(columns=to_drop)

    return X2, to_drop

def drop_quasi_constant_cat(
    X: pd.DataFrame, 
    cat_cols: List[str], 
    p: float = 0.99
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop quasi-constant categorical columns from a DataFrame.

    This function identifies categorical columns where the most frequent value 
    accounts for at least `p` proportion of the data and removes them from the DataFrame. 
    Quasi-constant columns are those that provide little variability and are unlikely 
    to be useful for modeling.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    cat_cols : List[str]
        A list of categorical column names to consider for quasi-constant checking.
    p : float, optional
        The proportion threshold above which a column is considered quasi-constant.
        Default is 0.99.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with quasi-constant categorical columns removed.
        - A list of the names of the dropped quasi-constant categorical columns.

    Notes:
    -----
    - The function ensures that only columns present in the DataFrame are processed.
    - Missing values are included in the value counts when determining the most frequent value.

    Example Usage:
    --------------
    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_cat_cols = drop_quasi_constant_cat(X, cat_cols.tolist(), p=0.99)
    print(f"Dropped quasi-constant categorical columns: {dropped_cat_cols}")
    """
    dropped = []
    for c in cat_cols:
        # Calculate the normalized value counts (including NaNs)
        vc = X[c].value_counts(normalize=True, dropna=False)
        
        # Check if the most frequent value exceeds the threshold
        if len(vc) and vc.iloc[0] >= p:
            dropped.append(c)
    
    # Drop the identified quasi-constant columns
    X2 = X.drop(columns=dropped)
    return X2, dropped

def drop_high_cardinality_cat(
    X: pd.DataFrame, 
    cat_cols: List[str], 
    max_unique: int = 100, 
    ratio: float = 0.50
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop high-cardinality categorical columns from a DataFrame.

    This function identifies categorical columns with a high number of unique values 
    (cardinality) and removes them from the DataFrame. High-cardinality columns can 
    increase the complexity of the model and may not provide significant value.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    cat_cols : List[str]
        A list of categorical column names to consider for cardinality checking.
    max_unique : int, optional
        The maximum number of unique values allowed in a categorical column. 
        Default is 100.
    ratio : float, optional
        The maximum ratio of unique values to the total number of rows in the DataFrame.
        Default is 0.50.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-cardinality categorical columns removed.
        - A list of the names of the dropped high-cardinality categorical columns.

    Notes:
    -----
    - A column is considered high-cardinality if the number of unique values exceeds 
      `max_unique` or if the ratio of unique values to the total number of rows exceeds `ratio`.
    - The function ensures that only columns present in the DataFrame are processed.

    Example Usage:
    --------------
    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_high_card_cols = drop_high_cardinality_cat(X, cat_cols.tolist(), max_unique=100, ratio=0.50)
    print(f"Dropped high-cardinality categorical columns: {dropped_high_card_cols}")
    """
    n = len(X)
    lim = min(max_unique, int(ratio * n))
    dropped = [c for c in cat_cols if c in X.columns and X[c].nunique(dropna=False) > lim]
    X2 = X.drop(columns=dropped)
    return X2, dropped

def drop_quasi_constant_num(
    X: pd.DataFrame, num_cols: List[str], thresh: float = 1e-5
) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop quasi-constant numeric columns from a DataFrame.

    This function identifies numeric columns with very low variance (below a specified threshold)
    and removes them from the DataFrame. Quasi-constant columns are those that have almost the same
    value across all rows, which makes them uninformative for modeling.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    num_cols : List[str]
        A list of numeric column names to consider for variance checking.
    thresh : float, optional
        The variance threshold below which columns are considered quasi-constant.
        Default is 1e-5.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with quasi-constant numeric columns removed.
        - A list of the names of the dropped quasi-constant numeric columns.

    Notes:
    -----
    - Missing values in the numeric columns are imputed using the median before calculating variance.
    - The VarianceThreshold from sklearn is used to identify quasi-constant columns.
    """
    if not num_cols:
        return X, []

    # Impute missing values with the median
    imp = SimpleImputer(strategy="median")
    Xn = pd.DataFrame(imp.fit_transform(X[num_cols]), columns=num_cols, index=X.index)

    # Apply VarianceThreshold to identify columns with low variance
    vt = VarianceThreshold(threshold=thresh)
    vt.fit(Xn)

    # Identify kept and dropped columns
    kept = [c for c, k in zip(num_cols, vt.get_support()) if k]
    dropped = [c for c in num_cols if c not in kept]

    # Drop the quasi-constant columns from the original DataFrame
    X2 = X.drop(columns=dropped)
    return X2, dropped

def prefilter_num_univariate(
    X: pd.DataFrame, y: pd.Series, num_cols: List[str], k: int = 300
) -> Tuple[pd.DataFrame, List[str], pd.Series]:
    """
    Perform a univariate feature selection for numeric columns based on their relationship with the target variable.

    This function selects the top `k` numeric features that have the highest scores in a univariate statistical test 
    (ANOVA F-value for classification or F-statistic for regression) with respect to the target variable `y`.

    Parameters:
    ----------
    X : pd.DataFrame
        The input dataframe containing features.
    y : pd.Series
        The target variable.
    num_cols : List[str]
        A list of numeric column names to consider for feature selection.
    k : int, optional
        The number of top features to keep, by default 300.

    Returns:
    -------
    Tuple[pd.DataFrame, List[str], pd.Series]
        - A dataframe containing the top `k` numeric features.
        - A list of the names of the selected top `k` numeric features.
        - A pandas Series containing the scores of all numeric features, sorted in descending order.

    Notes:
    -----
    - If the target variable `y` is numeric and has more than 20 unique values, the function uses `f_regression`.
      Otherwise, it uses `f_classif`.
    - Missing values in the numeric columns are imputed using the median before calculating the scores.
    """
    # Filter numeric columns that exist in the dataframe
    keep = [c for c in num_cols if c in X.columns]
    if not keep:
        return X, [], pd.Series(dtype=float)

    # Impute missing values with the median
    Xi = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X[keep]),
        columns=keep, index=X.index
    )

    # Select the appropriate statistical test based on the target variable type
    if np.issubdtype(y.dtype, np.number) and y.nunique() > 20:
        scores, _ = f_regression(Xi, y.to_numpy())
    else:
        scores, _ = f_classif(Xi, y.to_numpy())

    # Create a pandas Series of scores, sort them in descending order
    s = pd.Series(scores, index=keep).fillna(0.0).sort_values(ascending=False)

    # Select the top `k` features
    top = s.index[: min(k, len(s))].tolist()

    # Return the top features, their names, and the scores
    return pd.concat([Xi[top]], axis=1), top, s

def drop_zeros(X: pd.DataFrame, thresh: float = 0.95) -> Tuple[pd.DataFrame, List[str]]:
    """
    Drop columns with a high proportion of zero values from a DataFrame.

    This function identifies columns in the DataFrame where the proportion of zero values 
    exceeds a specified threshold and removes them. Columns with a high proportion of zeros 
    are often uninformative and can be excluded from further analysis.

    Parameters:
    ----------
    X : pd.DataFrame
        The input DataFrame containing the features.
    thresh : float, optional
        The proportion threshold above which a column is considered to have high zero values 
        and is dropped. Default is 0.95 (95%).

    Returns:
    -------
    Tuple[pd.DataFrame, List[str]]
        - A DataFrame with high-zero columns removed.
        - A list of the names of the dropped columns.

    Notes:
    -----
    - The function calculates the proportion of zero values in each column.
    - Columns with a proportion of zero values greater than `thresh` are dropped.

    Example Usage:
    --------------
    Xnum, dropped_cols = drop_zeros(Xnum, thresh=0.95)
    print(f"Dropped columns with >95% zeros: {dropped_cols}")
    """
    # Identify columns with a high proportion of zero values
    to_drop = X.columns[(X == 0).mean() > thresh].tolist()

    # Drop the identified columns
    X = X.drop(columns=to_drop)

    return X, to_drop


In [413]:
def clean_up(train,test):
    train.set_index('SamplingOperations_code')
    Xtr = train.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status'])).set_index('SamplingOperations_code')
    Y = train[['SamplingOperations_code', 'IBD', 'IBD_EQR', 'IBD_EQR_Status']].set_index('SamplingOperations_code')
    y = train[['IBD']]
    Xte=test.set_index('SamplingOperations_code')
    
    X = pd.concat([Xtr, Xte], axis=0)
    COMPLETE_X = X.copy()
    X = X.drop(columns=['Date_SamplingOperation'])

    """En este espacio es dodne podemos hacer limpieza de datos """
    X, dropped_dup_cols = drop_exact_duplicates(X)
    print(f"Dropped exact duplicate columns: {dropped_dup_cols}")

    X, dropped_cols = drop_high_missing(X, thresh=0.95)
    print(f"Dropped columns with >95% missing: {dropped_cols}")
    print(len(dropped_cols))

    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_cat_cols = drop_quasi_constant_cat(X, cat_cols.tolist(), p=0.99)
    print(f"Dropped quasi-constant categorical columns: {dropped_cat_cols}")

    cat_cols = X.select_dtypes(exclude=['number']).columns
    X, dropped_high_card_cols = drop_high_cardinality_cat(X, cat_cols.tolist(), max_unique=100, ratio=0.50)
    print(f"Dropped high-cardinality categorical columns: {dropped_high_card_cols}")

    num_cols = X.select_dtypes(include=['number']).columns  # Select numeric columns
    X, dropped_num_cols = drop_quasi_constant_num(X, num_cols.tolist(), thresh=1e-5)
    print(f"Dropped quasi-constant numeric columns: {dropped_num_cols}")
    
    COMPLETE_X[dropped_cols]

    uncommon_taxons = COMPLETE_X[dropped_cols].sum(axis=1)
    uncommon_taxons = uncommon_taxons.rename("Uncommon_Taxons")

    df3 = pd.concat([X, uncommon_taxons], axis=1)

    df3 = df3.join(Y, how='left')

    

    return df3 

In [407]:
def get_region_sites_and_taxa(region, sites, taxones):
    """
    Regresa los dos DF que armaste a mano:
    - df: subset de sites para la región
    - df1: subset de taxones filtrado por los SamplingOperations_code válidos de df
    """
    df = sites[sites['HERlvl1Code'] == region].copy()
    valid_codes = df['SamplingOperations_code'].unique()
    df1 = taxones[taxones['SamplingOperations_code'].isin(valid_codes)].copy()
    return  df1


In [408]:
def pivot_taxa_abundance_pm(df1):
    """
    Hace el pivote exactamente como en tu código:
    - index: todas las columnas salvo TaxonName, Abundance_nbcell, TaxonCode, Abundance_pm
    - columns: TaxonCode
    - values: Abundance_pm
    - aggfunc: sum
    """
    # columnas que conservas en la llave
    drop_cols = ['TaxonName', 'Abundance_nbcell', 'TaxonCode', 'Abundance_pm']
    cols_key = [c for c in df1.columns if c not in drop_cols]

    wide = (
        df1.pivot_table(index=cols_key,
                        columns='TaxonCode',
                        values='Abundance_pm',
                        aggfunc='sum')
           .reset_index()
    )
    wide.columns.name = None
    return wide


In [409]:
train

,SamplingOperations_code,IBD,IBD_EQR,IBD_EQR_Status
0,S02000008_20170703,9.6,0.502924,Poor
1,S02000008_20200708,8.0,0.409357,Poor
2,S02000010_20070906,14.3,0.777778,Moderate
3,S02000010_20090721,15.0,0.818713,Good
4,S02000010_20110723,16.0,0.877193,Good
...,...,...,...,...
43563,S06999141_20160928,15.6,0.706667,Moderate
43564,S06999179_20230628,20.0,1.000000,High
43565,S06999180_20160928,19.5,0.966667,High
43566,S06999189_20160928,16.1,0.740000,Moderate


In [410]:
def build_region_train_test(region, sites, taxones, pressure, train, test_codes, fillna_with=0):
    """
    Flujo completo por región:
    1) Filtra sites y taxones (df, df1)
    2) Pivotea abundancias permil por TaxonCode (wide)
    3) Une con pressure (df2) usando las mismas llaves que tú pusiste
    4) Saca train_df (inner contra 'train' por SamplingOperations_code)
    5) Saca test_df filtrando df2 por test_codes

    Parámetros:
      - region: int/str con el código de región (HERlvl1Code)
      - sites, taxones, pressure, train: DataFrames existentes
      - test_codes: iterable de SamplingOperations_code para test
      - fillna_with: si quieres rellenar NaN post-pivote (0 por default). Usa None para no rellenar.

    Regresa:
      - train_df, test_df
    """
    # 1) df y df1
    df1 = get_region_sites_and_taxa(region, sites, taxones)

    # Si no hay datos para la región, devuelve vacíos con mismas columnas.
    if df1.empty:
        empty_train = pd.DataFrame(columns=['SamplingOperations_code'])
        empty_test  = pd.DataFrame(columns=['SamplingOperations_code'])
        return empty_train, empty_test

    # 2) pivote
    wide = pivot_taxa_abundance_pm(df1)

    # 3) merge con pressure (exactamente tus llaves)
    df2 = pd.merge(
        wide,
        pressure,
        on=['SamplingOperations_code', 'CodeSite_SamplingOperations', 'Date_SamplingOperation'],
        how='inner'
    )

   
    df3 = pd.merge(df2, train, on='SamplingOperations_code', how='left')    
    # 5) test_df (filtrado por test_codes)
    col = 'IBD'   # columna en dfB que checas

    mask = df3[col].isna()          # True si NaN

    train_df = df3.loc[~mask]
    test_df = df3.loc[mask]  
    test_df=test_df.drop(columns=(['IBD', 'IBD_EQR', 'IBD_EQR_Status']))

    cleandf = clean_up(train= train_df,test=test_df)

    return cleandf


In [414]:
# Ejemplo de uso:
region = 21
cleandf= build_region_train_test(
    region=region,
    sites=sites,
    taxones=taxones,
    pressure=pressure,
    train=train,
    test_codes=test_codes,   # asegúrate de tenerlo definido
    fillna_with=0            # pon None si no quieres rellenar
)


Dropped exact duplicate columns: ['Pinrh02']
Dropped columns with >95% missing: ['Achaa01', 'Achac01', 'Achaf01', 'Achaf02', 'Achal01', 'Achan01', 'Achba01', 'Achbi02', 'Achca03', 'Achca04', 'Achco01', 'Achco02', 'Achcr01', 'Achde02', 'Achde03', 'Achdi01', 'Achdi02', 'Achdr01', 'Achel01', 'Achen01', 'Achex01', 'Achex02', 'Achfr01', 'Achfu01', 'Achge01', 'Achgr02', 'Achgr03', 'Achhe01', 'Achho01', 'Achhu01', 'Achja02', 'Achjo01', 'Achko01', 'Achku01', 'Achla03', 'Achla04', 'Achla06', 'Achle02', 'Achli01', 'Achli02', 'Achli03', 'Achlu02', 'Achma01', 'Achmi03', 'Achna02', 'Achne01', 'Achno01', 'Achpa01', 'Achpe02', 'Achpf01', 'Achps04', 'Achpu01', 'Achpy01', 'Achre01', 'Achro01', 'Achro02', 'Achru01', 'Achsa01', 'Achse02', 'Achsi01', 'Achst01', 'Achst03', 'Achsu06', 'Achsu07', 'Achte01', 'Achtr02', 'Achtr03', 'Achzh01', 'Achzi01', 'Actde01', 'Actno01', 'Adlba01', 'Adlbr01', 'Adlbr02', 'Adlmu02', 'Adlpa01', 'Adlsu01', 'Ampat01', 'Ampei01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampov

In [ ]:
def crear_modelo(cleandf):
    cleandf = cleandf[cleandf['IBD'].notna()]
    X = cleandf.drop(columns=['IBD','IBD_EQR','IBD_EQR_Status'])
    y = cleandf['IBD']
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    # 2) Identify column types
    num_cols = X.select_dtypes(include=['number']).columns
    cat_cols = X.columns.difference(num_cols)
    # 3) Preprocess
    # this preprocessor can handle missing values   
    pre = ColumnTransformer([
        # numerical features
        ('num', Pipeline([
            # imputation and scaling
            ('imp', SimpleImputer(strategy='median')),
            # scaling (RF doesn't need it, but other models might)
            # ('scaler', StandardScaler(with_mean=False))  # stays sparse with OHE
        ]), num_cols),
        # categorical features
        ('cat', Pipeline([
            # imputation as None being another category
            ('imp', SimpleImputer(strategy='constant', fill_value='None')),
            # one-hot encoding, ignoring unknown categories during inference
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), cat_cols)

    ])
    # 4) Model
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, random_state=42, ))  # for regression
    ])  
    clf.fit(X_tr, y_tr)
    print("R2 train:", clf.score(X_tr, y_tr))
    print("R2 valid:", clf.score(X_te, y_te))

    return clf

In [417]:
clf = crear_modelo(cleandf)

R2 train: 0.9790034187053251
R2 valid: 0.8288111692690423


In [419]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def train_region_model(cleandf, target='IBD'):
    # 1) separar train y scoring dentro de la MISMA región
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols)
    y = df_train[target].astype(float)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo
    clf = Pipeline([
        ('pre', pre),
        ('model', RandomForestRegressor(n_estimators=600, 
                                        random_state=42, 
                                        n_jobs=-1))
    ])

    # 4) hold-out (barajado dentro de la región)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

    clf.fit(X_tr, y_tr)
    pred_tr = clf.predict(X_tr)
    pred_te = clf.predict(X_te)

    metrics = {
        'R2_train': r2_score(y_tr, pred_tr),
        'R2_valid': r2_score(y_te, pred_te),
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
    }

    # 5) (opcional) KFold simple dentro de la región
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    cv_r2 = cross_val_score(clf, X, y, cv=kf, scoring='r2', n_jobs=-1)
    metrics['R2_CV_mean'] = cv_r2.mean()
    metrics['R2_CV_all']  = cv_r2

    # 6) predecir filas sin target de esta región
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        preds_score = clf.predict(X_score)
        scored = df_score.copy()
        scored[target + '_pred'] = preds_score
    else:
        scored = pd.DataFrame(columns=cleandf.columns.tolist() + [target + '_pred'])

    return clf, metrics, scored


In [420]:
model, metrics, scored = train_region_model(cleandf, target='IBD')
print(metrics)
# scored trae las filas SIN IBD con la columna IBD_pred


{'R2_train': 0.9790856340204913, 'R2_valid': 0.8289758937124094, 'MAE_train': 0.2700774798927676, 'MAE_valid': 0.7792530335474669, 'RMSE_train': 0.14444726322609894, 'RMSE_valid': 1.2640867709374266, 'R2_CV_mean': np.float64(0.8384223048157505), 'R2_CV_all': array([0.82755961, 0.83745028, 0.83044998, 0.85299148, 0.84366017])}


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor

def train_catboost_region(
    cleandf: pd.DataFrame,
    target: str = 'IBD',
    test_size: float = 0.20,
    random_state: int = 42,
    early_stopping_rounds: int = 200,
    cat_params: dict | None = None,
):
    """
    Entrena CatBoost para una región usando cleandf.
    - Separa train (con target) y score (sin target)
    - Preprocesa (imputación num/cat + OHE)
    - Entrena con early stopping
    - Regresa: modelo (Pipeline), métricas, scored_df (filas sin target con predicción)
    """
    # 1) separar train / score
    df_train = cleandf[cleandf[target].notna()].copy()
    df_score = cleandf[cleandf[target].isna()].copy()

    # X / y
    drop_cols = [c for c in ['IBD','IBD_EQR','IBD_EQR_Status'] if c in cleandf.columns]
    X = df_train.drop(columns=drop_cols, errors='ignore')
    y = df_train[target].astype(float)

    # split
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=test_size, random_state=random_state)

    # 2) columnas num/cat
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

    # preprocesamiento
    pre = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
        ]), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='(missing)')),
            ('ohe', OneHotEncoder(handle_unknown='ignore')),
        ]), cat_cols)
    ])

    # 3) modelo CatBoost (parámetros por defecto + override opcional)
    base_params = dict(
        depth=6, learning_rate=0.05, n_estimators=3000,
        loss_function='RMSE', random_state=random_state, verbose=0
    )
    if cat_params:
        base_params.update(cat_params)

    cb = Pipeline([
        ('pre', pre),
        ('model', CatBoostRegressor(**base_params))
    ])

    # 4) ajustar: primero ajustamos el preprocesador para armar matrices, luego el modelo con early stopping
    Xtr_proc = pre.fit_transform(X_tr)
    Xte_proc = pre.transform(X_te)
    cb.named_steps['model'].fit(
        Xtr_proc, y_tr,
        eval_set=(Xte_proc, y_te),
        use_best_model=True,
        early_stopping_rounds=early_stopping_rounds
    )

    # 5) métricas en hold-out y train (usando el pipeline para que transforme igual)
    r2_tr  = cb.score(X_tr, y_tr)
    r2_te  = cb.score(X_te, y_te)
    pred_tr = cb.predict(X_tr); pred_te = cb.predict(X_te)
    metrics = {
        'R2_train': r2_tr,
        'R2_valid': r2_te,
        'MAE_train': mean_absolute_error(y_tr, pred_tr),
        'MAE_valid': mean_absolute_error(y_te, pred_te),
        'RMSE_train': mean_squared_error(y_tr, pred_tr),
        'RMSE_valid': mean_squared_error(y_te, pred_te),
        'best_iterations': int(cb.named_steps['model'].get_best_iteration() or base_params['n_estimators'])
    }

    # 6) predicciones para filas sin target de esta región (si existen)
    if not df_score.empty:
        X_score = df_score.drop(columns=drop_cols, errors='ignore')
        df_score[target + '_pred'] = cb.predict(X_score)
        scored_df = df_score
        
    else:
        scored_df = pd.DataFrame(columns=list(cleandf.columns) + [target + '_pred'])

    return cb, metrics, scored_df


In [451]:
model, metrics, scored = train_catboost_region(cleandf, target='IBD')
print(metrics)
# 'scored' contiene las filas sin IBD con la columna IBD_pred


{'R2_train': np.float64(0.999990040561215), 'R2_valid': np.float64(0.9019932987946552), 'MAE_train': 0.005295542403706627, 'MAE_valid': 0.4890119412491184, 'RMSE_train': 4.2071701797490975e-05, 'RMSE_valid': 0.5385305720122008, 'best_iterations': 2995}


In [459]:
solo_ibd_cols = scored[['IBD_pred']].reset_index()  # ahora tienes columnas: SamplingOperations_code, IBD
solo_ibd_cols

,SamplingOperations_code,IBD_pred
0,S02000010_20080811,14.479493
1,S02000010_20100719,15.194359
2,S02000010_20150811,13.903476
3,S02000010_20160825,14.921369
4,S02000010_20170703,15.851655
...,...,...
182,S06457310_20160928,13.125038
183,S06458450_20200723,12.998203
184,S06471450_20160929,11.860209
185,S06471450_20180628,13.820255


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd

# carpetas de salida
os.makedirs("models/regions", exist_ok=True)
os.makedirs("preds/regions", exist_ok=True)

all_metrics = []
models      = {}
all_preds   = []  # para concatenar predicciones globales

KEY = 'SamplingOperations_code'
TARGET = 'IBD'

for region in regiones:
    # --- construir el cleandf de la región ---
    out = build_region_train_test(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )
    # si tu función devuelve varias cosas, toma el DF
    cleandf = out if isinstance(out, pd.DataFrame) else out[0]
    cdf = cleandf.copy()

    # asegura que la llave esté como columna
    if cdf.index.name == KEY and KEY not in cdf.columns:
        cdf = cdf.reset_index()

    # --- entrenar modelo CatBoost por región ---
    model, metrics, _scored = train_catboost_region(cdf, target=TARGET, key=KEY)

    # --- guardar modelo ---
    model_path = f"models/regions/catboost_region_{region}.joblib"
    joblib.dump(model, model_path)
    models[region] = model  # también en memoria

    # --- tamaños y features ---
    drop_cols = [c for c in [TARGET,'IBD_EQR','IBD_EQR_Status'] if c in cdf.columns]
    n_rows   = cdf.shape[0]
    n_train  = cdf[TARGET].notna().sum() if TARGET in cdf.columns else np.nan
    n_score  = cdf[TARGET].isna().sum() if TARGET in cdf.columns else np.nan
    n_feats  = cdf.drop(columns=drop_cols, errors='ignore').shape[1]

    # --- métricas ---
    row = {
        'region': region,
        'model_path': model_path,
        'n_rows': n_rows,
        'n_train': n_train,
        'n_to_score': n_score,
        'n_features': n_feats
    }
    row.update(metrics)  # R2_train, R2_valid, MAE, RMSE, best_iterations, etc.
    all_metrics.append(row)

    # --- PREDICCIONES PARA TODAS LAS FILAS DE LA REGIÓN ---
    X_full = cdf.drop(columns=drop_cols, errors='ignore')
    # predice con el pipeline (transforma internamente)
    yhat_full = model.predict(X_full)

    preds_region = pd.DataFrame({
        KEY: cdf[KEY].values,
        'IBD_pred': yhat_full
    }).set_index(KEY).sort_index()

    # guarda SOLO predicciones, con índice = KEY
    scored_path = f"preds/regions/IBD_preds_region_{region}.parquet"
    preds_region.to_parquet(scored_path, index=True)

    # acumula para el archivo global
    tmp = preds_region.copy()
    tmp['region'] = region
    all_preds.append(tmp)

# --- DataFrames finales globales ---
results_df = pd.DataFrame(all_metrics).sort_values('region').reset_index(drop=True)
results_df.to_csv("models/region_metrics.csv", index=False)

if all_preds:
    preds_df = pd.concat(all_preds, axis=0)
    # deja índice = KEY; region queda como columna
    preds_df.to_parquet("preds/IBD_preds_all_regions.parquet", index=True)

print("Guardados:")
print("  - métricas: models/region_metrics.csv")
print("  - modelos por región: models/regions/*.joblib")
print("  - predicciones por región (solo IBD_pred, index=SamplingOperations_code): preds/regions/*.parquet")
print("  - predicciones globales: preds/IBD_preds_all_regions.parquet")


In [ ]:
predicciones = []
all_metrics = {}  # guardará {region: métricas}

for region in regiones:
    cleandf = build_region_train_test(
        region=region,
        sites=sites,
        taxones=taxones,
        pressure=pressure,
        train=train,
        test_codes=test_codes,
        fillna_with=0
    )
    model, m, scored = train_catboost_region(cleandf, target='IBD')  # m = métricas de esa región
    all_metrics[region] = m

    # Mantener índice (SamplingOperations_code) y solo la predicción
    solo_ibd = scored[['IBD_pred']].copy()
    solo_ibd['region'] = region
    predicciones.append(solo_ibd)

predicciones = pd.concat(predicciones, axis=0)  # índice preservado
predicciones.index.name = 'SamplingOperations_code'
    

Dropped exact duplicate columns: ['Gompr03', 'Navkr01', 'Schex01']
Dropped columns with >95% missing: ['Achaf01', 'Achaf02', 'Achat01', 'Achca02', 'Achca04', 'Achch01', 'Achco01', 'Achda01', 'Achda02', 'Achde02', 'Achdr01', 'Achex01', 'Achex02', 'Achfu01', 'Achgr02', 'Achgr03', 'Achhi01', 'Achho01', 'Achho03', 'Achhu01', 'Achja02', 'Achko01', 'Achkr02', 'Achla03', 'Achla06', 'Achli03', 'Achob01', 'Achpf01', 'Achpu01', 'Achre01', 'Achro02', 'Achru01', 'Achth01', 'Achtr03', 'Achzh01', 'Adlbr02', 'Adlmi01', 'Adlsu01', 'Ampco01', 'Ampma01', 'Ampme01', 'Ampmi02', 'Ampmo01', 'Ampne01', 'Ampne02', 'Ampno01', 'Amppe01', 'Ampve01', 'Ampve02', 'Astfo01', 'Aulbr01', 'Auldi01', 'Aulps01', 'Aulsu01', 'Aulsu02', 'Bacpa01', 'Bacul01', 'Berru01', 'Bible01', 'Brane01', 'Brane02', 'Breke01', 'Calal01', 'Calfo01', 'Calsc01', 'Calsi01', 'Cocdi01', 'Cochu01', 'Cocne02', 'Cocne03', 'Cocpa01', 'Cocps02', 'Cocse01', 'Conwe01', 'Cosla02', 'Ctepu01', 'Cycat01', 'Cycco01', 'Cyccy01', 'Cycde02', 'Cycdi01', 'Cyckr